In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Relatório-sem-título-nov-25-2025-a-dez-4-2025.csv")

df.columns = [
    "anuncio", "criativo", "alcance", "impressoes", "frequencia",
    "moeda", "valor_usado", "atribuicao", "cliques_link",
    "cpc", "cpm", "engajamento", "conversas_mensagem",
    "custo_por_conversa", "ctr", "inicio", "fim"
]

def limpa_dinheiro(serie):
    return (
        serie.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

df['valor_usado'] = (
    df['valor_usado']
    .astype(str)
    .str.replace("R$", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

for col in ['cpc', 'cpm', 'custo_por_conversa']:
    df[col] = (
        df[col].astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

df['ctr'] = (
    df['ctr']
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float) / 100
)

numeric_cols = [
    "alcance", "impressoes", "frequencia", "cliques_link",
    "engajamento", "conversas_mensagem"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['inicio'] = pd.to_datetime(df['inicio'], errors='coerce')
df['fim'] = pd.to_datetime(df['fim'], errors='coerce')

df['cpc_calc'] = df['valor_usado'] / df['cliques_link'].replace(0, np.nan)
df['cpm_calc'] = df['valor_usado'] / df['impressoes'].replace(0, np.nan) * 1000
df['ctr_calc'] = df['cliques_link'] / df['impressoes'].replace(0, np.nan)
df['custo_por_conversa_calc'] = df['valor_usado'] / df['conversas_mensagem'].replace(0, np.nan)

df_formatado = df.copy()

def formata_reais(x):
    if pd.isna(x):
        return "-"
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

df_formatado['valor_usado'] = df_formatado['valor_usado'].apply(formata_reais)
df_formatado['cpc'] = df_formatado['cpc'].apply(formata_reais)
df_formatado['cpm'] = df_formatado['cpm'].apply(formata_reais)
df_formatado['custo_por_conversa'] = df_formatado['custo_por_conversa'].apply(formata_reais)
df_formatado['ctr'] = df_formatado['ctr'].apply(lambda x: "-" if pd.isna(x) else f"{x*100:.2f}%")

colunas_visual = [
    "criativo", "alcance", "impressoes", "frequencia",
    "valor_usado", "cliques_link", "cpc", "cpm",
    "engajamento", "conversas_mensagem", "custo_por_conversa", "ctr"
]

print(df_formatado[colunas_visual])


            criativo  alcance  impressoes  frequencia  valor_usado  \
0                NaN    36205       60289    1.665212  R$ 1.000,00   
1  VIDEO APRESETANDO    21681       27888    1.286288    R$ 363,04   
2  VIDEO APRESETANDO     9485       17585    1.853980    R$ 413,87   
3           VIDEO 03     4491        7091    1.578936    R$ 126,10   
4           VIDEO 02     2834        4288    1.513056     R$ 73,16   
5           VIDEO 03     2317        2641    1.139836     R$ 18,51   
6           VIDEO 02      681         796    1.168869      R$ 5,32   

   cliques_link      cpc       cpm  engajamento  conversas_mensagem  \
0          1650  R$ 0,61  R$ 16,59        15840                91.0   
1          1339  R$ 0,27  R$ 13,02         9550                13.0   
2           205  R$ 2,02  R$ 23,54         3772                60.0   
3            35  R$ 3,60  R$ 17,78         1232                12.0   
4            25  R$ 2,93  R$ 17,06          906                 6.0   
5            

In [ ]:
linha_total = df_formatado.iloc[0].copy()
linha_total['criativo'] = 'TOTAL'

tabela_criativos = df_formatado.iloc[1:].reset_index(drop=True)

colunas_meta = [
    "criativo",
    "alcance",
    "impressoes",
    "frequencia",
    "valor_usado",
    "cliques_link",
    "cpc",
    "cpm",
    "engajamento",
    "conversas_mensagem",
    "custo_por_conversa",
    "ctr"
]

print("TOTAL DA CAMPANHA:")
print(linha_total[colunas_meta])

print("\nCRIATIVOS INDIVIDUAIS:")
tabela_criativos[colunas_meta]


TOTAL DA CAMPANHA:
criativo                    TOTAL
alcance                     36205
impressoes                  60289
frequencia               1.665212
valor_usado           R$ 1.000,00
cliques_link                 1650
cpc                       R$ 0,61
cpm                      R$ 16,59
engajamento                 15840
conversas_mensagem           91.0
custo_por_conversa       R$ 10,99
ctr                         3.37%
Name: 0, dtype: object

CRIATIVOS INDIVIDUAIS:


,criativo,alcance,impressoes,frequencia,valor_usado,cliques_link,cpc,cpm,engajamento,conversas_mensagem,custo_por_conversa,ctr
0,VIDEO APRESETANDO,21681,27888,1.286288,"R$ 363,04",1339,"R$ 0,27","R$ 13,02",9550,13.0,"R$ 27,93",5.01%
1,VIDEO APRESETANDO,9485,17585,1.853980,"R$ 413,87",205,"R$ 2,02","R$ 23,54",3772,60.0,"R$ 6,90",2.27%
2,VIDEO 03,4491,7091,1.578936,"R$ 126,10",35,"R$ 3,60","R$ 17,78",1232,12.0,"R$ 10,51",1.49%
3,VIDEO 02,2834,4288,1.513056,"R$ 73,16",25,"R$ 2,93","R$ 17,06",906,6.0,"R$ 12,19",1.80%
4,VIDEO 03,2317,2641,1.139836,"R$ 18,51",35,"R$ 0,53","R$ 7,01",257,NaN,-,1.44%
5,VIDEO 02,681,796,1.168869,"R$ 5,32",11,"R$ 0,48","R$ 6,68",123,NaN,-,1.51%
